# Домашнее задание: Pandas + визуализация (Facebook Ads)
Этот ноутбук выполняет все пункты задания (1–6) по файлу **facebook_ads_data.csv**.

⚠️ Если файл называется иначе (например, `facebook_ads_data (2.0).csv`), код ниже попробует найти его автоматически.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# (опционально) seaborn понадобится только для lmplot
import seaborn as sns

# --- загрузка данных ---
candidates = [
    "facebook_ads_data.csv",
    "facebook_ads_data (2.0).csv",
    "/mnt/data/facebook_ads_data.csv",
    "/mnt/data/facebook_ads_data (2.0).csv",
]
path = next((p for p in candidates if os.path.exists(p)), "facebook_ads_data.csv")

df = pd.read_csv(path)
df.head()

## Подготовка
ROMI будем считать как `total_value / total_spend` (если `total_spend` = 0 → NaN). Поле `ad_date` оставляем строкой — фильтрация по датам сработает и так (по подсказке в задании).

In [ ]:
# ROMI (на уровне строк)
df["romi_calc"] = np.where(df["total_spend"] > 0, df["total_value"] / df["total_spend"], np.nan)

# Быстрая проверка
df[["ad_date","campaign_name","total_spend","total_value","romi","romi_calc"]].head()

## 1) Группировка по дням за 2021 год: расходы и дневной ROMI + rolling mean

In [ ]:
df_2021 = df[(df["ad_date"] >= "2021-01-01") & (df["ad_date"] < "2022-01-01")].copy()

daily_2021 = (
    df_2021
    .groupby("ad_date", as_index=False)
    .agg(
        total_spend=("total_spend", "sum"),
        total_value=("total_value", "sum"),
    )
    .sort_values("ad_date")
)

daily_2021["romi_daily"] = np.where(daily_2021["total_spend"] > 0, daily_2021["total_value"] / daily_2021["total_spend"], np.nan)

# rolling mean (например, 7 дней)
window = 7
daily_2021["spend_rm"] = daily_2021["total_spend"].rolling(window=window, min_periods=1).mean()
daily_2021["romi_rm"]  = daily_2021["romi_daily"].rolling(window=window, min_periods=1).mean()

daily_2021.head()

In [ ]:
# График: дневная сумма расходов (2021) + скользящее среднее
plt.figure(figsize=(12, 4))
plt.plot(daily_2021["ad_date"], daily_2021["total_spend"], label="Daily total_spend")
plt.plot(daily_2021["ad_date"], daily_2021["spend_rm"], label=f"Rolling mean ({window}d)")
plt.title("Daily total_spend (2021)")
plt.xlabel("ad_date")
plt.ylabel("total_spend")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# График: дневной ROMI (2021) + скользящее среднее
plt.figure(figsize=(12, 4))
plt.plot(daily_2021["ad_date"], daily_2021["romi_daily"], label="Daily ROMI")
plt.plot(daily_2021["ad_date"], daily_2021["romi_rm"], label=f"Rolling mean ({window}d)")
plt.title("Daily ROMI (2021)")
plt.xlabel("ad_date")
plt.ylabel("ROMI")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 2) Группировка по названию кампании: суммарные расходы и общий ROMI

In [ ]:
by_campaign = (
    df
    .groupby("campaign_name", as_index=False)
    .agg(
        total_spend=("total_spend", "sum"),
        total_value=("total_value", "sum"),
    )
)

by_campaign["romi_total"] = np.where(by_campaign["total_spend"] > 0, by_campaign["total_value"] / by_campaign["total_spend"], np.nan)

# Удобно отсортировать по расходам
by_campaign = by_campaign.sort_values("total_spend", ascending=False)
by_campaign

In [ ]:
# График: суммарные расходы по кампаниям
plt.figure(figsize=(10, 5))
plt.bar(by_campaign["campaign_name"], by_campaign["total_spend"])
plt.title("Total spend by campaign")
plt.xlabel("campaign_name")
plt.ylabel("total_spend")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# График: общий ROMI по кампаниям
plt.figure(figsize=(10, 5))
plt.bar(by_campaign["campaign_name"], by_campaign["romi_total"])
plt.title("Total ROMI by campaign")
plt.xlabel("campaign_name")
plt.ylabel("ROMI")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3) Box plot: разброс дневного ROMI в каждой кампании

In [ ]:
# Сначала посчитаем дневной ROMI внутри каждой кампании
daily_by_campaign = (
    df
    .groupby(["campaign_name", "ad_date"], as_index=False)
    .agg(
        total_spend=("total_spend", "sum"),
        total_value=("total_value", "sum"),
    )
)
daily_by_campaign["romi_daily"] = np.where(daily_by_campaign["total_spend"] > 0, daily_by_campaign["total_value"] / daily_by_campaign["total_spend"], np.nan)

daily_by_campaign.head()

In [ ]:
# Box plot (matplotlib)
# Подготовим данные списком по кампаниям в одном порядке
campaigns = daily_by_campaign["campaign_name"].unique()
data = [daily_by_campaign.loc[daily_by_campaign["campaign_name"] == c, "romi_daily"].dropna().values for c in campaigns]

plt.figure(figsize=(12, 6))
plt.boxplot(data, labels=campaigns, showfliers=True)
plt.title("Daily ROMI distribution by campaign (box plot)")
plt.xlabel("campaign_name")
plt.ylabel("daily ROMI")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4) Гистограмма распределения ROMI

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["romi_calc"].dropna(), bins=50)
plt.title("ROMI distribution (histogram)")
plt.xlabel("ROMI")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 5) Тепловая карта корреляций между числовыми показателями
Ниже — корреляция (Pearson) для всех числовых колонок. На этом датасете максимальная корреляция (вне диагонали) между **total_spend** и **total_value**: **0.979**. Минимальная — между **cpc** и **ctr**: **-0.211**.

Корреляции `total_value` с другими метриками (убывание):

- `total_value`: 1.000
- `total_spend`: 0.979
- `total_clicks`: 0.472
- `total_impressions`: 0.472
- `cpm`: 0.471
- `cpc`: 0.251
- `romi`: -0.014
- `ctr`: -0.022

In [ ]:
# Корреляционная матрица и тепловая карта
num = df.select_dtypes(include=[np.number])
corr = num.corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.title("Correlation heatmap (numeric columns)")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.tight_layout()
plt.show()

corr

## 6) Scatter + линейная регрессия: связь total_spend и total_value
В задании предложено `lmplot()`. Ниже — вариант с `seaborn.lmplot()` (и альтернативно можно сделать регрессию через `numpy.polyfit`, если нужно).

In [ ]:
# Seaborn lmplot (удобно для визуализации регрессии)
plot_df = df[["total_spend", "total_value"]].copy()

# Чтобы график был читабельнее, исключим нулевые расходы (опционально)
plot_df = plot_df[plot_df["total_spend"] > 0]

sns.lmplot(data=plot_df, x="total_spend", y="total_value", height=5, aspect=1.3)
plt.title("Linear regression: total_spend vs total_value")
plt.tight_layout()
plt.show()

### Готово ✅
Запусти все ячейки сверху вниз, убедись что графики строятся без ошибок, и отправляй `.ipynb` в форму сдачи.